In [ ]:
!pip install transformers accelerate sentencepiece

from transformers import pipeline
import datetime
from dataclasses import dataclass
import pandas as pd

# Load FLAN-T5 model
llm = pipeline("text-generation", model="google/flan-t5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [ ]:
@dataclass
class Assignment:
    title: str
    description: str
    due_date: datetime.date
    difficulty: int
    estimated_hours: float = 0
    priority: float = 0
    llm_suggestion: str = ""
    daily_plan: list = None

# ANSI color codes for readability
class Colors:
    HEADER = "\033[95m"
    BLUE = "\033[94m"
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    END = "\033[0m"

def priority_score(difficulty, days_left):
    """Higher difficulty + fewer days = higher priority."""
    if days_left <= 0:
        return 10
    return round((difficulty * 2) + (5 / days_left), 2)


In [ ]:
def llm_study_suggestion(description: str) -> str:
    prompt = (
        f"Give a short, practical study strategy for this assignment in 2–3 sentences: "
        f"{description}. Keep it clear, simple, and realistic for a college student."
    )
    result = llm(prompt, max_new_tokens=80)[0]["generated_text"]
    cleaned = result.replace("\n", " ").strip()
    return cleaned


In [ ]:
def estimate_time(description: str, difficulty: int) -> float:
    """
    Simple estimation model:
    - Base hours from difficulty
    - Add small boost for long descriptions
    """
    base = difficulty * 1.5
    length_factor = len(description.split()) / 40
    return round(base + length_factor, 2)


In [ ]:
def build_schedule(assignment: Assignment, start_date: datetime.date):
    days_left = (assignment.due_date - start_date).days

    # Priority score
    assignment.priority = priority_score(assignment.difficulty, days_left)

    # Color selection based on priority
    if assignment.priority >= 8:
        color = Colors.RED
    elif assignment.priority >= 6:
        color = Colors.YELLOW
    else:
        color = Colors.GREEN

    # If due today or overdue
    if days_left <= 0:
        assignment.daily_plan = [
            f"{color}⚠ URGENT: Complete all {assignment.estimated_hours} hours TODAY (past/near due).{Colors.END}"
        ]
        return

    hours_per_day = assignment.estimated_hours / days_left
    assignment.daily_plan = []

    for i in range(days_left):
        day = start_date + datetime.timedelta(days=i)
        assignment.daily_plan.append(
            f"{color}{day.isoformat()} — Work {hours_per_day:.2f} hrs on '{assignment.title}' "
            f"(Priority {assignment.priority}){Colors.END}"
        )


In [ ]:
def run_studyflow():
    print(f"{Colors.BOLD}{Colors.BLUE}Welcome to StudyFlow AI!{Colors.END}")

    num = int(input("How many assignments do you want to enter? "))
    assignments = []

    for i in range(num):
        print(f"\n--- Assignment {i+1} ---")
        title = input("Title: ")
        desc = input("Description: ")
        due = input("Due date (YYYY-MM-DD): ")
        difficulty = int(input("Difficulty (1–5): "))

        due_date = datetime.date.fromisoformat(due)
        assignment = Assignment(title, desc, due_date, difficulty)

        # LLM + time estimation
        assignment.estimated_hours = estimate_time(desc, difficulty)
        assignment.llm_suggestion = llm_study_suggestion(desc)

        # Build schedule
        today = datetime.date.today()
        build_schedule(assignment, today)

        assignments.append(assignment)

    # Output results
    print(f"\n\n{Colors.BOLD}{Colors.HEADER}===== STUDYFLOW AI RESULTS ====={Colors.END}")

    for i, a in enumerate(assignments):
        print(f"\n{Colors.BOLD}{Colors.BLUE}Assignment {i+1}: {a.title}{Colors.END}")
        print(f"{Colors.CYAN}Estimated Hours:{Colors.END} {a.estimated_hours}")
        print(f"{Colors.CYAN}Priority Score:{Colors.END} {a.priority}")
        print(f"{Colors.CYAN}LLM Study Suggestion:{Colors.END} {a.llm_suggestion}\n")

        print(f"{Colors.BOLD}Daily Plan:{Colors.END}")
        for day_plan in a.daily_plan:
            print("  -", day_plan)


In [ ]:
run_studyflow()

Welcome to StudyFlow AI!
How many assignments do you want to enter? 3

--- Assignment 1 ---
Title: English Paper
Description: MLA cited Paper 5 page maximum
Due date (YYYY-MM-DD): 2026-04-28
Difficulty (1–5): 3


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Assignment 2 ---
Title: Create LAN network
Description: Use Cisco packet tracer to create this network
Due date (YYYY-MM-DD): 2026-04-27
Difficulty (1–5): 1


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Assignment 3 ---
Title: Spanish Excerpt
Description: Write about a topic you enjoy completely in Spanish.  Must be 3 pages 
Due date (YYYY-MM-DD): 2026-05-10
Difficulty (1–5): 5


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




===== STUDYFLOW AI RESULTS =====

Assignment 1: English Paper
Estimated Hours: 4.65
Priority Score: 11.0
LLM Study Suggestion: Give a short, practical study strategy for this assignment in 2–3 sentences: MLA cited Paper 5 page maximum. Keep it clear, simple, and realistic for a college student.oncerning the subject concerned accordingthe student: “Bonfire from me

Daily Plan:
  - 2026-04-27 — Work 4.65 hrs on 'English Paper' (Priority 11.0)

Assignment 2: Create LAN network
Estimated Hours: 1.7
Priority Score: 10
LLM Study Suggestion: Give a short, practical study strategy for this assignment in 2–3 sentences: Use Cisco packet tracer to create this network. Keep it clear, simple, and realistic for a college student. entering/offlines/devance orders via router routes (i-1) into either Ethernet route. Encode as to all physical routing lines; donor packet trace router gateway rules such mail mailbox

Daily Plan:
  - ⚠ URGENT: Complete all 1.7 hours TODAY (past/near due).

Assignment 3: 